In [ ]:
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community sentence-transformers transformers faiss-cpu langchain-openai

Using Python 3.13.12 environment at: C:\Users\medha\Downloads\end to end RAG sy\.venv
Checked 6 packages in 22ms


In [79]:
from dotenv import load_dotenv
from pathlib import Path
import os

# Go to parent folder and find .env
env_path = Path.cwd() / ".." / ".env"

print("cwd:", Path.cwd())
print("env file:", env_path.resolve())
print("exists:", env_path.exists())

load_dotenv(env_path)

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

print("OPENAI loaded:", os.getenv("OPENAI_API_KEY") is not None)

cwd: c:\Users\medha\Downloads\end to end RAG sy\notebook
env file: C:\Users\medha\Downloads\end to end RAG sy\.env
exists: True
OPENAI loaded: True


In [80]:

from langchain.document_loaders import TextLoader

In [81]:

loader = TextLoader(r"C:\Users\medha\Downloads\end to end RAG sy\data\agenticaitopictext.txt", encoding="utf8")
documents = loader.load()

In [82]:

documents[0].page_content[:500] 

'Understanding Agentic AI\n\nAgentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.\n\nKey Characteristics of Agentic AI\n\nAgentic AI systems are distinct from traditional AI models due t'

In [83]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)
text_chunks=text_splitter.split_documents(documents)
text_chunks

[Document(metadata={'source': 'C:\\Users\\medha\\Downloads\\end to end RAG sy\\data\\agenticaitopictext.txt'}, page_content='Understanding Agentic AI'),
 Document(metadata={'source': 'C:\\Users\\medha\\Downloads\\end to end RAG sy\\data\\agenticaitopictext.txt'}, page_content='Agentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving'),
 Document(metadata={'source': 'C:\\Users\\medha\\Downloads\\end to end RAG sy\\data\\agenticaitopictext.txt'}, page_content='towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.'),
 Document(metadata={'source': 'C:\\Users\\medha\\Downloads\\end to end RAG sy\\data\\agenticaitopictext.txt'}, page_content='Key Characteristics of Agentic AI\n\nAgentic AI systems are distinct from 

In [84]:
! uv pip install faiss-cpu

Using Python 3.13.12 environment at: C:\Users\medha\Downloads\end to end RAG sy\.venv
Checked 1 package in 227ms


In [88]:
from langchain.vectorstores import FAISS

try:
    from langchain.embeddings import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    print("✓ Using local sentence-transformers embeddings")
except Exception as e:
    print("⚠ Could not use local HuggingFace embeddings:", e)
    print("Falling back to OpenAIEmbeddings")
    from langchain.embeddings import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2006.10it/s]


✓ Using local sentence-transformers embeddings


In [89]:

vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [90]:
from dotenv import load_dotenv
from pathlib import Path
import os

root = Path.cwd()
load_dotenv(root / ".." / ".env")

openrouter_key = os.getenv("YOUR_OPENROUTER_API_KEY")
if not openrouter_key:
    raise RuntimeError("YOUR_OPENROUTER_API_KEY is not set in .env")

from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough

llm_model = ChatOpenAI(
    model="deepseek/deepseek-chat",
    base_url="https://openrouter.ai/api/v1",
    api_key=openrouter_key,
    temperature=0.7,
)
print("✓ Using Deepseek via OpenRouter")

template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.

Question: {question}

Context:
{context}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)
retriever = vectorstore.as_retriever()
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm_model
)

response = rag_chain.invoke("Tell me about Agentic AI")
print(response)

✓ Using Deepseek via OpenRouter
content='Agentic AI represents a new paradigm in artificial intelligence where systems operate autonomously to achieve goals rather than just responding to queries. It focuses on self-directed decision-making and adaptability across various tasks. The potential applications of Agentic AI span numerous industries, making it a transformative technology. Its future involves advancing autonomy, improving efficiency, and enabling more complex problem-solving. Agentic AI is designed to go beyond traditional AI by taking proactive actions. This approach enhances scalability and reduces the need for constant human oversight. Industries like healthcare, finance, and logistics could benefit significantly from its capabilities. The development of Agentic AI aims to create systems that learn and evolve independently. Its growth could redefine how AI integrates into daily operations. Understanding Agentic AI is key to leveraging its full potential.' additional_kwargs